In [1]:
%pip install -q torchmetrics
%pip install -q optuna

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports
import random

import numpy as np
import optuna

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy

In [3]:
# Reproducibilty
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # For CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(45)
device = ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
def get_data():
    # Load dataset and convert labels to 0-based indexing
    data = fetch_covtype(random_state=45)
    x, y = data.data, data.target - 1

    # --- Preprocessing ---
    # Split dataset into train, val, test
    x_train_full, x_test, y_train_full, y_test = train_test_split(x, y, stratify=y,
                                                                  test_size=0.2, random_state=45)
    x_train, x_val, y_train, y_val = train_test_split(x_train_full, y_train_full, stratify=y_train_full,
                                                      test_size=0.15, random_state=45)

    # Scale dataset (standard scaler)
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_val_scaled = scaler.transform(x_val)

    # Delete redundant datasets
    del x_train_full, y_train_full, x_train, x_val, x_test, y_test

    # Convert to PyTorch tensors
    x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    x_val_tensor = torch.tensor(x_val_scaled, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)

    # Create TensorDataset object
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(x_val_tensor, y_val_tensor)

    # Create Dataloader
    g = torch.Generator()   #for deterministic DataLoader shuffling
    g.manual_seed(45)
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [ ]:
# Training & Evaluation Functions
def train_one_epoch(model, data_loader, optimizer, device):
    model.train()
    for input, target in data_loader:
        input, target = input.to(device), target.to(device)

        # Forward
        logits = model(input)
        loss = F.cross_entropy(logits, target)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Validation
def evaluate(model, data_loader, accuracy_metric, device):
    model.eval()
    with torch.no_grad():
        for input, target in data_loader:
              input, target = input.to(device), target.to(device)
              logits = model(input)
              preds = torch.argmax(logits, dim=1)

              accuracy_metric.update(preds, target)

        overall_accuracy = accuracy_metric.compute()
    return overall_accuracy.item()

# Instantiate metric
val_accuracy = Accuracy(task='multiclass', num_classes=7).to(device)

In [6]:
# Optuna
def objective(trial):
    # hyperparameters to tune
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    num_hidden_layer = trial.suggest_int("num_hidden_layer", 1, 3)

    input_dim, output_dim = 54, 7

    layers = []
    current_input_dim = input_dim
    for i in range(num_hidden_layer):
        n_units = trial.suggest_int(f"n_units{i}", 32, 128, step =16)
        layers.append(nn.Linear(current_input_dim, n_units))
        layers.append(nn.ReLU())
        current_input_dim = n_units

    layers.append(nn.Linear(current_input_dim, output_dim))
    model = nn.Sequential(*layers).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    train_loader, val_loader = get_data()

    # Train & evaluate
    num_epochs = 25
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        train_one_epoch(model, train_loader, optimizer, device)
        val_acc = evaluate(model, val_loader, val_accuracy, device)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return best_val_acc


sampler = optuna.samplers.TPESampler(seed = 45)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3, interval_steps =1)

study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials = 50)

# Best result
print(f"Best trial:\n{study.best_trial.params}")
print(f"Best value: {study.best_trial.value}")

[I 2026-03-30 19:55:54,239] A new study created in memory with name: no-name-21ec1208-362f-4f08-ba0d-2cb6d7172c81
[I 2026-03-30 20:04:23,631] Trial 0 finished with value: 0.8356736898422241 and parameters: {'learning_rate': 0.09269035391921815, 'num_hidden_layer': 2, 'n_units0': 48, 'n_units1': 32}. Best is trial 0 with value: 0.8356736898422241.
[I 2026-03-30 20:11:46,381] Trial 1 finished with value: 0.8306064009666443 and parameters: {'learning_rate': 0.002154807546828439, 'num_hidden_layer': 2, 'n_units0': 32, 'n_units1': 48}. Best is trial 0 with value: 0.8356736898422241.
[I 2026-03-30 20:19:04,109] Trial 2 finished with value: 0.7960378527641296 and parameters: {'learning_rate': 0.0002227676556617142, 'num_hidden_layer': 2, 'n_units0': 112, 'n_units1': 96}. Best is trial 0 with value: 0.8356736898422241.
[I 2026-03-30 20:26:23,858] Trial 3 finished with value: 0.7920987606048584 and parameters: {'learning_rate': 0.09379183965007398, 'num_hidden_layer': 2, 'n_units0': 96, 'n_unit

Best trial:
{'learning_rate': 0.09269035391921815, 'num_hidden_layer': 2, 'n_units0': 48, 'n_units1': 32}
Best value: 0.8356736898422241
